

**Author:** Lionel Cedric Gohouede

# IMPORT LIBRARIES

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
from numba import njit
import math
import xarray as xr
import matplotlib.pyplot as plt
from google.colab import drive
from matplotlib.dates import DateFormatter
from scipy.optimize import minimize

##USE DATA FROM PYTHON

In [ ]:
pip install aqua-fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.1/289.1 kB 5.5 MB/s eta 0:00:00


In [ ]:
from aqua_fetch import RainfallRunoff

rr = RainfallRunoff("CAMELS_FR")

/usr/local/lib/python3.12/dist-packages/aqua_fetch/rr/utils.py:127: UserWarning: netCDF4 module is not installed. Please install it to save data in netcdf format
  warnings.warn(msg, UserWarning)


downloading 6 files to /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
downloading ADDITIONAL_LICENSES.zip
0% of 0.07 MB downloaded
100% of 0.07 MB downloaded
downloading CAMELS_FR_attributes.zip
0% of 9.88 MB downloaded
100% of 9.88 MB downloaded
downloading CAMELS_FR_geography.zip
0% of 1.45 MB downloaded
100% of 1.45 MB downloaded
downloading CAMELS_FR_time_series.zip
0% of 361.39 MB downloaded
20% of 361.39 MB downloaded
40% of 361.39 MB downloaded
60% of 361.39 MB downloaded
80% of 361.39 MB downloaded
100% of 361.39 MB downloaded
downloading README.md
0% of 0.01 MB downloaded
100% of 0.01 MB downloaded
downloading CAMELS-FR_description.ods
0% of 0.05 MB downloaded
100% of 0.05 MB downloaded
unzipping files in /usr/local/lib/python3.12/dist-packages/aqua_fetch/data/CAMELS/CAMELS_FR
unzipping CAMELS_FR_attributes.zip to CAMELS_FR_attributes
unzipping ADDITIONAL_LICENSES.zip to ADDITIONAL_LICENSES
unzipping CAMELS_FR_time_series.zip to CAMELS_FR_time_series


In [ ]:
meta, ds = rr.fetch()

Read 654 stations for 22 dyn features in 58.40 seconds with 2 cpus.


In [ ]:
print(ds.head())

<xarray.Dataset> Size: 131kB
Dimensions:           (time: 5, dynamic_features: 5)
Coordinates:
  * time              (time) datetime64[ns] 40B 1970-01-01 ... 1970-01-05
  * dynamic_features  (dynamic_features) object 40B 'q_cms_obs' ... 'tsd_val_m'
Data variables: (12/654)
    A105003001        (time, dynamic_features) float64 200B 1.65e+03 ... 10.0
    A107020001        (time, dynamic_features) float64 200B nan nan ... nan nan
    A112020001        (time, dynamic_features) float64 200B 1.04e+03 ... 10.0
    A116003002        (time, dynamic_features) float64 200B nan nan ... nan nan
    A140202001        (time, dynamic_features) float64 200B nan nan ... nan nan
    A202030001        (time, dynamic_features) float64 200B nan nan ... nan nan
    ...                ...
    Y661401001        (time, dynamic_features) float64 200B 860.0 0.441 ... 10.0
    Y781000101        (time, dynamic_features) float64 200B nan nan ... nan nan
    Y862000101        (time, dynamic_features) float64 200B 1.

In [ ]:
print(ds.dynamic_features.values)

['q_cms_obs' 'q_mm_obs' 'tsd_val_s' 'tsd_val_q' 'tsd_val_m' 'tsd_val_c'
 'tsd_val_i' 'pcp_mm' 'pcp_mm_solfrac' 'airtemp_C_mean' 'pet_mm_ou'
 'pet_mm_pe' 'pet_mm_pm' 'windspeed_mps' 'spechum_gkg' 'lwdownrad_wm2'
 'solrad_wm2' 'tsd_swi_gr' 'tsd_swi_isba' 'tsd_swe_isba' 'airtemp_C_min'
 'airtemp_C_max']


In [ ]:
# Full period
print(ds["time"].values[0], ds["time"].values[-1])

1970-01-01T00:00:00.000000000 2021-12-31T00:00:00.000000000


###General

**Functions**

In [ ]:
import math

# ============================================
# dHyMoLAP Model
# ============================================
@njit
def dHyMoLAP_Model(params, Q0, q):
    mu, lambda_, Qs, qs = params
    N = len(q)

    k = np.zeros(N)
    x = np.zeros(N)
    r = np.zeros(N)

    if qs > 0.0:
        for t in range(N):
            r[t] = q[t] / qs

    if Qs > 0.0:
        k[0] = Q0 / Qs
    else:
        k[0] = 0.0

    if lambda_ == 0.0:
        return np.zeros(N)

    mu_over_lambda = mu / lambda_
    one_minus_mu_over_lambda = 1.0 - mu_over_lambda
    one_over_lambda = 1.0 / lambda_
    power_term = 2.0 * mu - 1.0

    for t in range(1, N):

        if math.isnan(r[t]):
            k[t] = k[t-1]
            x[t] = x[t-1]
            continue

        if r[t] > 0.0:
            x[t] = x[t-1] + mu_over_lambda * r[t]
        else:
            x[t] = one_minus_mu_over_lambda * x[t-1]

        k_prev = k[t-1]
        if k_prev <= 0.0:
            decay = 0.0
        else:
            decay = mu_over_lambda * (k_prev ** power_term)

        k[t] = max(0.0, k_prev - decay + one_over_lambda * x[t] * r[t])

    return k * Qs

# ============================================
# Metrics (pure NumPy, masked on valid pairs only)
# ============================================
def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    valid_obs = obs[mask]
    valid_sim = sim[mask]

    if valid_obs.size == 0 or np.var(valid_obs) == 0.0:
        return np.nan

    numerator = np.sum((valid_sim - valid_obs) ** 2)
    denominator = np.sum((valid_obs - np.mean(valid_obs)) ** 2)
    return 1.0 - (numerator / denominator)

def RMSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    valid_obs = obs[mask]
    valid_sim = sim[mask]

    if valid_obs.size == 0:
        return np.nan

    return np.sqrt(np.mean((valid_sim - valid_obs) ** 2))

def NNSE(obs, sim):
    nse = NSE(obs, sim)
    if not np.isfinite(nse):
        return np.nan
    return 1.0 / (2.0 - nse)

def PBIAS(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    valid_obs = obs[mask]
    valid_sim = sim[mask]

    if valid_obs.size == 0:
        return np.nan

    sum_obs = np.sum(valid_obs)
    if sum_obs == 0.0:
        return np.nan

    return np.sum(valid_sim - valid_obs) / sum_obs

def FHV(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o = obs[mask]
    s = sim[mask]
    L = o.size

    if L == 0:
        return np.nan

    order_desc = np.argsort(o)[::-1]
    n_hv = max(1, int(round(0.02 * L)))
    idx_hv = order_desc[:n_hv]
    sum_obs_hv = np.sum(o[idx_hv])

    if sum_obs_hv == 0.0:
        return np.nan

    return np.sum(s[idx_hv] - o[idx_hv]) / sum_obs_hv

def FLV(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o = obs[mask]
    s = sim[mask]
    L = o.size

    if L == 0:
        return np.nan

    order_asc = np.argsort(o)
    n_lv = max(1, int(round(0.30 * L)))
    idx_lv = order_asc[:n_lv]
    sum_obs_lv = np.sum(o[idx_lv])

    if sum_obs_lv == 0.0:
        return np.nan

    return np.sum(s[idx_lv] - o[idx_lv]) / sum_obs_lv

def KGE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o = obs[mask]
    s = sim[mask]

    if o.size == 0:
        return np.nan

    obs_mean = np.mean(o)
    sim_mean = np.mean(s)
    sigma_obs = np.std(o)
    sigma_sim = np.std(s)

    if sigma_obs == 0.0 or sigma_sim == 0.0 or obs_mean == 0.0:
        return np.nan

    r = np.corrcoef(o, s)[0, 1]
    if not np.isfinite(r):
        return np.nan

    alpha = sigma_sim / sigma_obs
    beta = sim_mean / obs_mean

    return 1.0 - np.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)

# ============================================
# Fonction objectif
# ============================================
def objective(params, Q0, q_train, Q_obs_train):
    Q_sim = dHyMoLAP_Model(params, Q0, q_train)
    nse = NSE(Q_obs_train, Q_sim)
    return 1.0 - nse if np.isfinite(nse) else 1e9

## **Main Code**

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit
import warnings

# Suppress runtime warnings for expected nan operations
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ============================================
# 1. COMPILED MODEL & FAST METRICS
# ============================================
@njit
def dHyMoLAP_Model(params, Q0, q):
    mu, lambda_, Qs, qs = params[0], params[1], params[2], params[3]
    N = len(q)
    k = np.zeros(N)
    x = np.zeros(N)

    if Qs <= 0.0 or qs <= 0.0 or lambda_ <= 0.0 or mu <= 0.0:
        return np.full(N, np.nan)

    k[0] = Q0 / Qs
    mu_lam = mu / lambda_
    one_m_mu_lam = 1.0 - mu_lam
    inv_lam = 1.0 / lambda_
    pow_term = 2.0 * mu - 1.0

    for t in range(N - 1):
        r_next = q[t+1] / qs
        if np.isnan(r_next):
            k[t+1] = k[t]
            x[t+1] = x[t]
            continue

        if r_next > 0.0:
            x[t+1] = x[t] + mu_lam * r_next
        else:
            x[t+1] = one_m_mu_lam * x[t]

        k_base = max(0.0, k[t])
        k[t+1] = max(0.0, k[t] - mu_lam * (k_base ** pow_term) + inv_lam * x[t+1] * r_next)

    return k * Qs

def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    valid_obs, valid_sim = obs[mask], sim[mask]
    if valid_obs.size == 0 or np.var(valid_obs) == 0.0:
        return np.nan
    return 1.0 - (np.sum((valid_sim - valid_obs)**2) / np.sum((valid_obs - np.mean(valid_obs))**2))

def compute_all_metrics(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o, s = obs[mask], sim[mask]
    L = o.size

    if L == 0:
        return dict(NSE=np.nan, NNSE=np.nan, RMSE=np.nan, PBIAS=np.nan, FHV=np.nan, FLV=np.nan, KGE=np.nan)

    obs_mean, sim_mean = np.mean(o), np.mean(s)
    denom = np.sum((o - obs_mean) ** 2)

    nse = np.nan if denom == 0.0 else 1.0 - np.sum((s - o) ** 2) / denom
    nnse = 1.0 / (2.0 - nse) if np.isfinite(nse) else np.nan
    rmse = np.sqrt(np.mean((s - o) ** 2))

    sum_obs = np.sum(o)
    pbias = np.sum(s - o) / sum_obs if sum_obs != 0.0 else np.nan

    order_desc = np.argsort(o)[::-1]
    n_hv = max(1, int(round(0.02 * L)))
    sum_obs_hv = np.sum(o[order_desc[:n_hv]])
    fhv = np.sum(s[order_desc[:n_hv]] - o[order_desc[:n_hv]]) / sum_obs_hv if sum_obs_hv != 0.0 else np.nan

    order_asc = np.argsort(o)
    n_lv = max(1, int(round(0.30 * L)))
    sum_obs_lv = np.sum(o[order_asc[:n_lv]])
    flv = np.sum(s[order_asc[:n_lv]] - o[order_asc[:n_lv]]) / sum_obs_lv if sum_obs_lv != 0.0 else np.nan

    sigma_obs, sigma_sim = np.std(o), np.std(s)
    if sigma_obs == 0.0 or sigma_sim == 0.0 or obs_mean == 0.0:
        kge = np.nan
    else:
        r = np.corrcoef(o, s)[0, 1]
        kge = np.nan if not np.isfinite(r) else 1.0 - np.sqrt((r - 1.0) ** 2 + (sigma_sim / sigma_obs - 1.0) ** 2 + (sim_mean / obs_mean - 1.0) ** 2)

    return dict(NSE=nse, NNSE=nnse, RMSE=rmse, PBIAS=pbias, FHV=fhv, FLV=flv, KGE=kge)

# ============================================
# 2. OPTIMIZATION OBJECTIVES FOR ABLATION
# ============================================
def obj_base(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], p[2], p[3]], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_qs(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], p[2], 1.0], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_Qs(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], 1.0, p[2]], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

def obj_fix_both(p, Q0, q, obs):
    nse = NSE(obs, dHyMoLAP_Model(np.array([p[0], p[1], 1.0, 1.0], dtype=np.float64), Q0, q))
    return 1.0 - nse if np.isfinite(nse) else 1e9

# ============================================
# 3. RUNTIME EXECUTION
# ============================================
ds_recent = ds.sel(time=slice("2000-01-01", "2021-12-31")).load()
all_stations = list(ds_recent.keys())

b1_ratio = 0.7
max_missing_ratio = 0.1

MU_BOUNDS = (0.5, 5.0)
LAMBDA_BOUNDS = (1e-3, 200.0)

results_base = {}
results_fix_qs = {}
results_fix_Qs = {}
results_fix_both = {}

for station_id in all_stations:
    Q_obs_raw = ds_recent[station_id].sel(dynamic_features="q_cms_obs").to_numpy()
    Q_obs = Q_obs_raw / 1000.0  # Converted to cms if originally in L/s
    P = ds_recent[station_id].sel(dynamic_features="pcp_mm").to_numpy()
    PET = ds_recent[station_id].sel(dynamic_features="pet_mm_pm").to_numpy()

    q = np.maximum(0, P - PET)
    N = len(Q_obs)

    if N == 0 or np.all(np.isnan(Q_obs)) or (np.sum(np.isnan(Q_obs)) / N) > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)
    valid_Q_idx = np.where(~np.isnan(Q_obs))[0]
    Q0 = Q_obs[valid_Q_idx[0]] if len(valid_Q_idx) > 0 else 0.0

    q_train = q[:b1]
    Q_obs_train = Q_obs[:b1]

    Qs_upper = 2.0 * np.nanmax(Q_obs_train) if np.any(~np.isnan(Q_obs_train)) else np.nan
    qs_upper = 2.0 * np.nanmax(q_train) if np.any(~np.isnan(q_train)) else np.nan

    if not np.isfinite(Qs_upper) or not np.isfinite(qs_upper) or Qs_upper <= 1e-3 or qs_upper <= 1e-3:
        continue

    Qs_mean = np.clip(np.nanmean(Q_obs_train), 1e-3, Qs_upper)
    qs_mean = np.clip(np.nanmean(q_train), 1e-3, qs_upper)
    opt_opts = {'maxiter': 2500, 'disp': False}

    # === Configuration 1: Base (Optimize qs and Qs) ===
    res1 = minimize(obj_base, np.array([1.1, 20.0, Qs_mean, qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, Qs_upper), (1e-3, qs_upper)], options=opt_opts)
    p_base = np.array(res1.x, dtype=np.float64)

    # === Configuration 2: Fix qs = 1 ===
    res2 = minimize(obj_fix_qs, np.array([1.1, 20.0, Qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, Qs_upper)], options=opt_opts)
    p_fix_qs = np.array([res2.x[0], res2.x[1], res2.x[2], 1.0], dtype=np.float64)

    # === Configuration 3: Fix Qs = 1 ===
    res3 = minimize(obj_fix_Qs, np.array([1.1, 20.0, qs_mean]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS, (1e-3, qs_upper)], options=opt_opts)
    p_fix_Qs = np.array([res3.x[0], res3.x[1], 1.0, res3.x[2]], dtype=np.float64)

    # === Configuration 4: Fix both = 1 ===
    res4 = minimize(obj_fix_both, np.array([1.1, 20.0]), args=(Q0, q_train, Q_obs_train),
                    method="Nelder-Mead", bounds=[MU_BOUNDS, LAMBDA_BOUNDS], options=opt_opts)
    p_fix_both = np.array([res4.x[0], res4.x[1], 1.0, 1.0], dtype=np.float64)

    # Simulate and Extract Metrics for All Configurations
    configs = [
        (results_base, p_base),
        (results_fix_qs, p_fix_qs),
        (results_fix_Qs, p_fix_Qs),
        (results_fix_both, p_fix_both)
    ]

    for res_dict, params in configs:
        Qsim = dHyMoLAP_Model(params, Q0, q)
        m_tr = compute_all_metrics(Q_obs_train, Qsim[:b1])
        m_vl = compute_all_metrics(Q_obs[b1:], Qsim[b1:])

        res_dict[station_id] = {
            "NSE_train": m_tr["NSE"], "NSE_val": m_vl["NSE"],
            "NNSE_train": m_tr["NNSE"], "NNSE_val": m_vl["NNSE"],
            "RMSE_train": m_tr["RMSE"], "RMSE_val": m_vl["RMSE"],
            "PBIAS_train": m_tr["PBIAS"], "PBIAS_val": m_vl["PBIAS"],
            "FHV_train": m_tr["FHV"], "FHV_val": m_vl["FHV"],
            "FLV_train": m_tr["FLV"], "FLV_val": m_vl["FLV"],
            "KGE_train": m_tr["KGE"], "KGE_val": m_vl["KGE"]
        }

# ============================================
# 4. STATISTICAL REPORTING MODULE
# ============================================
def print_statistics(results_dict, label):
    print(f"\n{'='*70}\n {label} \n{'='*70}")

    if not results_dict:
        print("No valid basins processed for this configuration.")
        return

    metrics = [
        ("NSE_train", "NSE Training"), ("NSE_val", "NSE Validation"),
        ("NNSE_train", "NNSE Training"), ("NNSE_val", "NNSE Validation"),
        ("KGE_train", "KGE Training"), ("KGE_val", "KGE Validation"),
        ("RMSE_train", "RMSE Training"), ("RMSE_val", "RMSE Validation"),
        ("PBIAS_train", "PBIAS Training"), ("PBIAS_val", "PBIAS Validation"),
        ("FHV_train", "FHV Training"), ("FHV_val", "FHV Validation"),
        ("FLV_train", "FLV Training"), ("FLV_val", "FLV Validation")
    ]

    for key, name in metrics:
        vals = np.array([r[key] for r in results_dict.values()], dtype=float)

        # Determine if array holds exclusively NaNs
        if np.all(np.isnan(vals)):
            print(f"**{name:17s}** | All generated metrics are NaN")
        else:
            print(f"**{name:17s}** | "
                  f"Mean: {np.nanmean(vals):>6.3f} | "
                  f"Median: {np.nanmedian(vals):>6.3f} | "
                  f"Min: {np.nanmin(vals):>6.3f} | "
                  f"Max: {np.nanmax(vals):>6.3f} | "
                  f"5th: {np.nanpercentile(vals, 5):>6.3f} | "
                  f"95th: {np.nanpercentile(vals, 95):>6.3f}")

# Trigger Reporting
print_statistics(results_base, "1. BASELINE CALIBRATION (Optimized qs and Qs)")
print_statistics(results_fix_qs, "2. ABLATION PHASE II (Fixed qs = 1)")
print_statistics(results_fix_Qs, "3. ABLATION PHASE III (Fixed Qs = 1)")
print_statistics(results_fix_both, "4. ABLATION PHASE IV (Fixed both qs = 1 and Qs = 1)")


 1. BASELINE CALIBRATION (Optimized qs and Qs) 
**NSE Training     ** | Mean:  0.637 | Median:  0.643 | Min:  0.047 | Max:  0.877 | 5th:  0.453 | 95th:  0.809
**NSE Validation   ** | Mean:  0.649 | Median:  0.674 | Min: -0.361 | Max:  0.900 | 5th:  0.411 | 95th:  0.835
**NNSE Training    ** | Mean:  0.739 | Median:  0.737 | Min:  0.512 | Max:  0.890 | 5th:  0.647 | 95th:  0.840
**NNSE Validation  ** | Mean:  0.749 | Median:  0.754 | Min:  0.423 | Max:  0.909 | 5th:  0.629 | 95th:  0.858
**KGE Training     ** | Mean:  0.700 | Median:  0.706 | Min: -0.072 | Max:  0.896 | 5th:  0.524 | 95th:  0.850
**KGE Validation   ** | Mean:  0.689 | Median:  0.715 | Min: -0.181 | Max:  0.911 | 5th:  0.460 | 95th:  0.853
**RMSE Training    ** | Mean:  5.034 | Median:  1.827 | Min:  0.088 | Max: 339.923 | 5th:  0.253 | 95th: 17.737
**RMSE Validation  ** | Mean:  4.756 | Median:  1.702 | Min:  0.074 | Max: 331.626 | 5th:  0.255 | 95th: 17.046
**PBIAS Training   ** | Mean:  0.020 | Median:  0.007 | Min: 